# CS 3110/5990: Data Privacy
## Homework 4

In [ ]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1 (10 points)

Complete the definition of `dp_sum_capgain` below. Your definition should compute a differentially private sum of the "Capital Gain" column of the `adult` dataset, and have a total privacy cost of `epsilon`.

In [34]:
def dp_sum_capgain(epsilon):
    b = 100000
    clipped_sum = adult['Capital Gain'].clip(lower=0, upper=b).sum()
    return laplace_mech(clipped_sum, sensitivity=b, epsilon=epsilon)

dp_sum_capgain(1.0)

np.float64(34856457.37029761)

In [37]:
# TEST CASE for question 1

real_sum = adult['Capital Gain'].sum()
r1 = np.mean([pct_error(real_sum, dp_sum_capgain(0.1)) for _ in range(100)])
r2 = np.mean([pct_error(real_sum, dp_sum_capgain(1.0)) for _ in range(100)])
r3 = np.mean([pct_error(real_sum, dp_sum_capgain(10.0)) for _ in range(100)])

print("Average errors:", r1, r2, r3)

assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 10
assert r2 < 2
assert r3 < 0.2

Average errors: 2.91414058214857 0.26363626908246085 0.02916805526511612


## Question 2 (10 points)

In 2-5 sentences each, answer the following:

- What clipping parameter did you use in your definition of `dp_sum_capital`, and why?
- What was the sensitivity of the query you used in `dp_sum_capital`, and how is it bounded?
- Argue that your definition of `dp_sum_capital` has a total privacy cost of `epsilon`

I used a clipping parameter of 100000. I chose it because the maximum Capital Gain value is 99999 and since it is recommended to include 100% of the dataset, I set the bound to be the next value. 

The sensitivity of the used query equals the difference in upper and lower bounds (since it is a summation query) and equals 100000. 

The query has a privacy cost of epsilon because the privacy cost of the Laplace mechanism is epsilon and that is the only query used.

## Question 3 (10 points)

Complete the definition of `dp_avg_capgain` below. Your definition should compute a differentially private average (mean) of the "Capital Gain" column of the adult dataset, and have a **total privacy cost of epsilon**.

In [38]:
def dp_avg_capgain(epsilon):
    sum = dp_sum_capgain(epsilon=epsilon/2)
    count = laplace_mech(len(adult["Capital Gain"]), sensitivity=1.0, epsilon=epsilon/2)
    return sum/count

dp_avg_capgain(1.0)

np.float64(1082.9841052041006)

In [39]:
# TEST CASE for question 3

real_avg = adult['Capital Gain'].mean()
r1 = np.mean([pct_error(real_avg, dp_avg_capgain(0.1)) for _ in range(100)])
r2 = np.mean([pct_error(real_avg, dp_avg_capgain(1.0)) for _ in range(100)])
r3 = np.mean([pct_error(real_avg, dp_avg_capgain(10.0)) for _ in range(100)])

print("Average errors:", r1, r2, r3)

assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 20
assert r2 < 4
assert r3 < 0.4

Average errors: 5.624939366516101 0.5803397568459521 0.05259793062826706


## Question 4 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your definition of `dp_avg_capgain` has a total privacy cost of `epsilon`
- For sums and averages, which seems to be more important for accuracy - the value of the clipping parameter $b$ or the scale of the noise added? Why?
- Do you think the answer to the previous point will be true for every dataset? Why or why not?

dp_avg_capgain has a total privacy cost of epsilon since it consists of two queries. The privacy cost of each is epsilon/2 and summing that up we receive epsilon. 

Clipping parameter is usually more important because it sets sensitivity and as a result impacts the noise. If b is reasonable and covers all or most of the data, the noise becomes more important, otherwise we are not going to receive reasonable results if the clipping parameter cuts half of the dataset.

It will not work the same way for every dataset because datasets may have different scales, so the clipping parameter and the noise need to be adjusted for each dataset.

## Question 5 (20 points)

Write a function `auto_avg` that returns the differentially private average of a Pandas series `s`. Your function should automatically determine the clipping parameter `b`, and should enforce differential privacy for a **total privacy cost** of `epsilon`. You can assume that all values are non-negative (i.e. 0 or greater).

In [98]:

def auto_avg(s, epsilon):
    b = s.max()
    sum = laplace_mech(s.clip(lower=0, upper=b).sum(), sensitivity=b, epsilon=epsilon/2)
    count = laplace_mech(len(s), sensitivity=1.0, epsilon=epsilon/2)
    return sum/count



In [104]:
# TEST CASE for question 5
r1 = np.mean([pct_error(adult['Age'].mean(), auto_avg(adult['Age'], 1.0)) for _ in range(20)])
r2 = np.mean([pct_error(adult['Capital Gain'].mean(), auto_avg(adult['Capital Gain'], 1.0)) for _ in range(20)])
r3 = np.mean([pct_error(adult['fnlwgt'].mean(), auto_avg(adult['fnlwgt'], 1.0)) for _ in range(20)])

print('Average errors:', r1, r2, r3)
assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 1
assert r2 < 100
assert r3 < 1

Average errors: 0.013814245598102867 0.4979633007294558 0.05034074312568254


PS:no hardcode, no worries if doesnt match all colloms; need to make it find cliping par by its own 

## Question 6

In 2-5 sentences each, answer the following:

- Explain your strategy for implementing `auto_avg`
- Argue informally that your definition of `auto_avg` has a total privacy cost of `epsilon`
- Did your solution work well for all three example columns? If it did not work well on any of them, why not?
- When is your solution likely to *not* work well? (i.e. what properties does the data have to have, in order for your solution to not work well?)

Implementing auto_avg I used the same approach as in the previous exercises. I set the clipping parameter to be the maximum value of the dataset to include 100% of the data as it is said in the book. After that I found the clipped differentially private sum and count and then computed the average.

Similarly to the previous exercises, the auto_avg has 2 queries each with cost of epsilon/2, so summing that up we receive epsilon.

I applied the same method that I tested in the previous exercises and because of that did not have many problems implementing it here.

It is likely to show worse results on data with a large number of outliers or on small datasets, since in those cases the noise can change the result too much.